## Concept focus — Type narrowing and runtime checks

Type narrowing connects dynamic runtime behavior to static understanding. It helps a type checker prove what is safe after a guard, and it helps you write code that becomes clearer to both humans and tools.

```text
value: str | None
    |
if value is not None:
    v
 value is now known as str inside this block
```

### How to think about it
A guard does two jobs at once: it protects runtime behavior and communicates intent. Ask yourself what facts become true after each `if`, `isinstance`, or sentinel check, then write code so those facts are obvious.

### Visual references and further study
- [typing documentation](https://docs.python.org/3/library/typing.html)
- [mypy type narrowing docs](https://mypy.readthedocs.io/en/stable/type_narrowing.html)
- [PEP 647 — TypeGuard](https://peps.python.org/pep-0647/)
- [Pyright docs](https://microsoft.github.io/pyright/)

---

# Module 17 — Typing and Static Analysis

## Exercise 17.2 — Ten narrowing puzzles

For each: predict whether `mypy --strict` accepts it, and whether it can fail at
runtime. Those are two separate questions and both need an answer.
Run:  mypy --strict ex02_narrowing.py
      python ex02_narrowing.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. The basics, in modern syntax

In [ ]:
def greet(name: str, times: int = 1) -> str: ...

nums: list[int] = []                    # builtin generics, 3.9+
mapping: dict[str, list[int]] = {}
pair: tuple[str, int] = ("a", 1)        # a fixed 2-tuple
row: tuple[int, ...] = (1, 2, 3)        # a variable-length tuple
maybe: str | None = None                # union syntax, 3.10+

Do not write `List[int]`, `Dict[str, int]`, or `Optional[str]` in new code. They
are the pre-3.9 spellings, still valid and now noise. `ruff`'s `UP` rules rewrite
them for you.

**Accept the widest type, return the narrowest.**

In [ ]:
from collections.abc import Iterable, Sequence, Mapping

def total(values: Iterable[float]) -> float: ...     # any iterable works
def process(items: Sequence[str]) -> list[str]: ...  # needs len/indexing
def lookup(config: Mapping[str, int]) -> int: ...    # read-only dict

Taking `Iterable` means a list, tuple, set, generator, or dict-keys view all
work. Taking `list` rejects four of those for no reason. Returning `list` tells
the caller they can index it; returning `Iterable` makes them guess.

And a hard-won rule from Module 14: **if your function iterates its argument
twice, it must take `Sequence`, not `Iterable`.** The type is the contract, and
`Iterable` promises only one pass.

---

## Concept 3. The vocabulary worth knowing

In [ ]:
from typing import Any, Literal, Final, TypeAlias, Self, NoReturn, cast, overload

x: Final = 3.14                          # never reassigned
Mode = Literal["r", "w", "a"]            # exactly these three strings
UserId: TypeAlias = int                  # a readable alias

def fail(msg: str) -> NoReturn:          # never returns normally
    raise RuntimeError(msg)

class Builder:
    def add(self, x: int) -> Self:       # 3.11+: returns THIS subclass
        return self

**`Literal` is underused and excellent.** `mode: Literal["r", "w"]` catches
`mode="rw"` at check time, and a `match` over a `Literal` can be checked for
exhaustiveness.

**`Any` disables checking**, silently and infectiously — every expression
involving an `Any` becomes `Any`. Treat it as a `# type: ignore` with a wider
blast radius. Prefer `object` when you truly do not know: `object` is safe (you
must narrow before using it), whereas `Any` permits everything.

**`cast` does nothing at runtime.** It is an assertion to the checker that you
know better. Each one is a place you have taken responsibility for a bug the
checker can no longer find.

---

## Concept 6. `TypedDict`, `NewType`, `overload`

In [ ]:
from typing import TypedDict, NewType, overload

class UserRecord(TypedDict):
    id: int
    name: str
    email: NotRequired[str]        # 3.11+

UserId = NewType("UserId", int)    # a distinct type at check time, an int at runtime

def get_user(uid: UserId) -> UserRecord: ...
get_user(42)                        # error: int is not UserId
get_user(UserId(42))                # fine

`NewType` is how you stop passing an order id where a user id was expected —
both are `int` to Python and distinct to the checker, at zero runtime cost.

In [ ]:
@overload
def parse(raw: str, *, strict: Literal[True]) -> Config: ...
@overload
def parse(raw: str, *, strict: Literal[False]) -> Config | None: ...
def parse(raw: str, *, strict: bool = True) -> Config | None: ...

`overload` expresses "the return type depends on an argument value" — exactly
the shape Module 04 said to avoid in a signature. When you cannot avoid it
(often because you are typing someone else's API), this is how you describe it.

---

## Concept 8. Static types versus runtime validation

**They solve different problems and you need both.**

| | Static (mypy) | Runtime (Pydantic) |
|---|---|---|
| When | Before running | While running |
| Cost | Zero at runtime | Real, per object |
| Catches | Wrong types in *your* code | Wrong types in *incoming data* |
| Cannot catch | Bad JSON from a client | A bug in a branch never run |

In [ ]:
# a boundary: data you did not create
class UserIn(BaseModel):          # Pydantic: validates and coerces at runtime
    name: str
    age: int

# inside: data you did create
@dataclass(frozen=True)           # dataclass + mypy: checked statically, free
class User:
    name: str
    age: int

**Validate at the boundary, trust inside.** Once `UserIn` has parsed the
request, everything downstream can rely on `age` being an `int`, and mypy will
enforce that it stays one. Module 28 builds on this.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: The basics, in modern syntax
- Section 2: `Optional` is not "optional"
- Section 3: The vocabulary worth knowing
- Section 4: Generics
- Section 5: `Protocol`: structural typing
- Section 6: `TypedDict`, `NewType`, `overload`
- Section 7: Running the checkers
- Section 8: Static types versus runtime validation

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Literal, TypeGuard

---

## `User`

_User_

In [ ]:
@dataclass
class User:
    name: str
    email: str | None = None

---

## `q01`

_q01_

In [ ]:
def q01(user: User | None) -> str:
    return user.name

---

## `q02`

_q02_

In [ ]:
def q02(user: User | None) -> str:
    if user is None:
        return "anonymous"
    return user.name

---

## `q03`

_q03_

In [ ]:
def q03(user: User | None) -> str:
    if user:
        return user.name
    return "anonymous"

---

## `q04`

_q04_

In [ ]:
def q04(user: User | None, refresh: bool) -> str:
    if user is None:
        return "anonymous"
    if refresh:
        user = lookup(user.name)          # returns User | None
    return user.name

---

## `lookup`

_lookup_

In [ ]:
def lookup(name: str) -> User | None:
    return User(name)

---

## `has_email`

_has email_

In [ ]:
def has_email(user: User) -> bool:
    return user.email is not None

---

## `q05`

_q05_

In [ ]:
def q05(user: User) -> str:
    if has_email(user):
        return user.email.upper()
    return ""

---

## `has_email_guard`

_has email guard_

In [ ]:
def has_email_guard(user: User) -> TypeGuard[User]:
    return user.email is not None

---

## `q07`

_q07_

In [ ]:
def q07(value: object) -> int:
    assert isinstance(value, int)
    return value + 1

---

## `q08`

_q08_

In [ ]:
def q08(mode: Mode) -> str:
    if mode == "read":
        return "r"
    elif mode == "write":
        return "w"
    else:
        assert_never(mode)          # type: ignore[name-defined]

---

## `q09`

_q09_

In [ ]:
def q09(raw: Any) -> int:
    parsed = raw["count"]           # Any
    doubled = parsed * 2            # still Any
    return doubled                  # returning Any from an int function

---

## `q10`

_q10_

In [ ]:
def q10() -> None:
    dogs: list[str] = ["rex"]
    animals: list[object] = dogs        # accepted?
    animals.append(42)                   # and then?
    print(dogs[1].upper())               # ...

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
# --- questions to answer -------------------------------------------------------
ANSWERS = """
q01  mypy:            runtime:
q02  mypy:            runtime:
q03  What does `if user:` actually narrow? Give a User subclass where `if user:`
     and `if user is not None:` differ. (Hint: Module 09.)
q04  Exactly which line loses the narrowing, and what is the fix that does NOT
     involve an assert?
q05  Why can mypy not follow has_email()?
q06  What does TypeGuard promise, and what happens if you LIE in one?
q07  Two separate questions: does mypy accept it, and what happens under
     python -O? (Module 01.)
q08  Add "append" to Mode. What error appears, and where? Now delete the
     assert_never line and re-check. What changed?
q09  Trace the Any. How many expressions became unchecked?
q10  Does mypy accept it? Should it? What is the general rule this
     demonstrates?
"""

if __name__ == "__main__":
    print(ANSWERS)

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.